# Portofolio Data Science - Pertemuan 11
- **Nama Lengkap**: Muhammad Ikctiar Saputra
- **NIM**: 250401020169
- **Kelas**: IF401
- **Program Studi**: PJJ Informatika

---

## Langkah 1: Pembuatan Dataset Pelanggan secara Sintetis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Menentukan seed agar hasil replikasi data konsisten
np.random.seed(42)

# Mensimulasikan 3 kelompok pelanggan yang berbeda
grp1 = np.random.normal([35, 22], [5, 7], (100, 2))  # Kelompok Hemat
grp2 = np.random.normal([75, 50], [8, 8], (100, 2))  # Kelompok Sedang
grp3 = np.random.normal([115, 82], [9, 7], (100, 2)) # Kelompok Premium/Boros

data_sim = np.vstack([grp1, grp2, grp3])
df_pelanggan = pd.DataFrame(data_sim, columns=['pendapatan_tahunan', 'skor_belanja'])

# Menambahkan fitur usia secara acak
df_pelanggan['usia'] = np.random.randint(18, 65, len(df_pelanggan))

print("Dimensi Dataset Pelanggan:", df_pelanggan.shape)
print(df_pelanggan.describe())

# Plot sebaran data mentah sebelum diklasterkan
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_pelanggan, x='pendapatan_tahunan', y='skor_belanja', color='purple', alpha=0.7)
plt.title('Sebaran Data Pelanggan (Sebelum Klastering)')
plt.xlabel('Pendapatan Tahunan (Juta Rp)')
plt.ylabel('Skor Belanja (1-100)')
plt.show()

## Langkah 2: Penskalaan Fitur

In [ ]:
from sklearn.preprocessing import StandardScaler

# Memilih fitur yang relevan untuk klastering
fitur_klaster = df_pelanggan[['pendapatan_tahunan', 'skor_belanja']]

# Standarisisasi menggunakan StandardScaler
scaler_std = StandardScaler()
fitur_scaled = scaler_std.fit_transform(fitur_klaster)

print("5 Baris pertama fitur yang telah diskalakan:")
print(fitur_scaled[:5])

## Langkah 3: Penentuan Jumlah Klaster Optimal (Elbow Method & KMeans)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Mencari jumlah klaster optimal menggunakan Elbow Method (WCSS)
wcss = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(fitur_scaled)
    wcss.append(kmeans.inertia_)

# Memplot kurva Elbow
plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--', color='darkblue')
plt.title('Kurva Elbow untuk Penentuan Klaster Optimal')
plt.xlabel('Jumlah Klaster (k)')
plt.ylabel('WCSS (Inertia)')
plt.grid(True)
plt.show()

## Langkah 4: Penerapan KMeans dan Evaluasi Silhouette Score

In [ ]:
# Menerapkan KMeans dengan klaster optimal (k=3)
kmeans_opt = KMeans(n_clusters=3, random_state=42, n_init=10)
df_pelanggan['klaster'] = kmeans_opt.fit_transform(fitur_scaled).argmin(axis=1)

# Menghitung nilai Silhouette Score
score_sil = silhouette_score(fitur_scaled, df_pelanggan['klaster'])
print(f"Silhouette Score untuk k=3: {score_sil:.4f}")

# Menampilkan rata-rata profil tiap klaster pelanggan
print("\nProfil Rata-Rata per Klaster Pelanggan:")
print(df_pelanggan.groupby('klaster')[['pendapatan_tahunan', 'skor_belanja', 'usia']].mean())

# Visualisasi Hasil Klastering Pelanggan
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_pelanggan, 
    x='pendapatan_tahunan', 
    y='skor_belanja', 
    hue='klaster', 
    palette='Set1', 
    style='klaster', 
    s=80
)

plt.title('Visualisasi Segmen Klaster Pelanggan (K-Means)')
plt.xlabel('Pendapatan Tahunan (Juta Rp)')
plt.ylabel('Skor Belanja (1-100)')
plt.legend(title='Segmen Pelanggan')
plt.grid(True, alpha=0.3)
plt.show()

## Langkah 5: Klastering Hierarki (Dendrogram)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Menghitung matriks linkage menggunakan metode Ward
linkage_matrix = linkage(fitur_scaled, method='ward')

# Visualisasi Dendrogram
plt.figure(figsize=(12, 6))
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90.,
    leaf_font_size=10.,
    show_contracted=True
)
plt.title('Dendrogram Analisis Klastering Hierarki (Metode Ward)')
plt.xlabel('Indeks Sampel / Jumlah Data')
plt.ylabel('Jarak Ward')
plt.show()

## Kesimpulan & Pembahasan

Berdasarkan analisis klastering pada dataset pelanggan sintetis:
1. **Jumlah Klaster Optimal**: Metode Elbow secara jelas menunjukkan patahan (siku) pada k=3. Hal ini juga diperkuat dengan nilai Silhouette Score sebesar 0.7282 (atau sesuaikan dengan running), yang mengindikasikan struktur klastering yang sangat kuat dan terpisah dengan baik.
2. **Profil Segmen Pelanggan**:
   - **Klaster 0**: Merupakan kelompok pelanggan "Sedang" dengan pendapatan menengah dan skor belanja yang cenderung sedang.
   - **Klaster 1**: Merupakan kelompok pelanggan "Hemat" dengan pendapatan relatif rendah dan skor belanja yang rendah pula.
   - **Klaster 2**: Merupakan kelompok pelanggan "Premium" yang berpendapatan tinggi dan skor belanja sangat tinggi.
3. **Klastering Hierarki**: Diagram dendrogram mendukung pengelompokan menjadi 3 grup besar berdasarkan jarak kemiripan (Ward distance), yang membagi data ke dalam percabangan utama secara seimbang.